In [4]:
# =============================================================================
# PIPELINE INTEGRACIÓN MULTI-PLATAFORMA – CÁNCER DE MAMA
# + ComBat (inmoose)
# + QC con PCA / UMAP / silhouette antes vs después
# =============================================================================
#
# INSTALACIÓN:
#   pip install inmoose pandas numpy scipy scikit-learn matplotlib seaborn umap-learn
#
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

# UMAP
try:
    import umap.umap_ as umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False


# =============================================================================
# ✏️ CONFIGURACIÓN
# =============================================================================

# ── Microarrays ──────────────────────────────────────────────────────────────
MICRO_MATRIX   = "./Data/expr_genes_microarrays.csv"
MICRO_METADATA = "./Data/microarrays_meta.csv"
MICRO_SAMPLE_COL = "sample"
MICRO_GSE_COL    = "batch"

# ── TCGA-BRCA ────────────────────────────────────────────────────────────────
TCGA_MATRIX   = "./Data/counts_symbol_TCGA.csv"
TCGA_METADATA = "./Data/TCGA_meta.csv"
TCGA_SAMPLE_COL = "sample"
TCGA_TYPE       = "log2"   # 'counts' | 'fpkm' | 'log2'

# ── GSE96058 ─────────────────────────────────────────────────────────────────
GSE96058_MATRIX   = "./Data/counts_symbol_GSE96058.csv"
GSE96058_METADATA = "./Data/GSE96058_meta.csv"
GSE96058_SAMPLE_COL = "sample"

# ── General ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("output_combat")
OUTPUT_DIR.mkdir(exist_ok=True)

# Covariable biológica a proteger en ComBat (si existe en metadata)
# p.ej. "label", "pam50_subtype", "subtype_label"
PROTECT_COL = "label"   # <- cámbialo a None si no quieres proteger nada

# Batch para ComBat
COMBAT_BATCH_COL = "dataset"   # "dataset" recomendado

# QC
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.3
UMAP_RANDOM_STATE = 42
PCA_N_COMPONENTS = 50


# =============================================================================
# UTILIDADES
# =============================================================================

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]
    return df


def detect_and_log2(df: pd.DataFrame, name: str) -> pd.DataFrame:
    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]
    vmax = np.percentile(vals, 99)
    vmin = np.min(vals)

    if vmax < 30 or vmin < 0:
        print(f"  [{name}] Escala log2 detectada (max p99={vmax:.1f}) → sin transformación")
    else:
        print(f"  [{name}] Escala lineal detectada (max p99={vmax:.0f}) → aplicando log2(x+1)")
        df = np.log2(df.clip(lower=0) + 1)
    return df


def ensure_unique_index(df: pd.DataFrame, axis_name: str = "index") -> pd.DataFrame:
    if df.index.duplicated().any():
        n_dup = int(df.index.duplicated().sum())
        print(f"  ⚠️ {n_dup} duplicados en {axis_name} → agregando por media")
        df = df.groupby(df.index).mean()
    return df


def align_samples(expr: pd.DataFrame, meta: pd.DataFrame,
                  sample_col: str, name: str):
    meta = clean_columns(meta)

    if sample_col not in meta.columns:
        raise ValueError(f"[{name}] No existe '{sample_col}' en metadata. Columnas: {list(meta.columns)}")

    expr = expr.copy()
    expr.columns = pd.Index(expr.columns.astype(str).str.strip())

    meta[sample_col] = meta[sample_col].astype(str).str.strip()

    if meta[sample_col].duplicated().any():
        n_dup = int(meta[sample_col].duplicated().sum())
        print(f"  [{name}] ⚠️ {n_dup} muestras duplicadas en metadata → se conserva la primera")
        meta = meta.drop_duplicates(subset=[sample_col], keep="first")

    meta = meta.set_index(sample_col, drop=False)

    common = expr.columns.intersection(meta.index)
    n_missing_expr = len(expr.columns) - len(common)
    n_missing_meta = len(meta.index) - len(common)

    if n_missing_expr > 0:
        print(f"  [{name}] ⚠️ {n_missing_expr} muestras en expresión sin metadata → excluidas")
    if n_missing_meta > 0:
        print(f"  [{name}] ⚠️ {n_missing_meta} muestras en metadata sin expresión → excluidas")

    expr_aligned = expr.loc[:, common]
    meta_aligned = meta.loc[common].copy()

    assert list(expr_aligned.columns) == list(meta_aligned.index), f"[{name}] desalineación expr/meta"
    return expr_aligned, meta_aligned

# =============================================================================
# EXTRA 1 – Guardar embeddings (PCA/UMAP) en CSV
# =============================================================================

def save_embeddings_csv(metadata, emb_pre, emb_post, out_dir=OUTPUT_DIR):
    """
    Guarda coordenadas PCA/UMAP antes y después de ComBat en CSV.
    Incluye metadata por muestra.
    """
    meta = metadata.copy()
    meta = meta.reindex(metadata.index)  # por claridad
    meta_out = meta.copy()

    # Asegura columna sample
    if "sample" not in meta_out.columns:
        meta_out["sample"] = meta_out.index.astype(str)

    # ---------- BEFORE ----------
    df_pre = meta_out.reset_index(drop=True).copy()

    # PCA (20 PCs)
    pca_pre = emb_pre.get("pca20", None)
    if pca_pre is not None:
        for i in range(pca_pre.shape[1]):
            df_pre[f"PCA{i+1}_pre"] = pca_pre[:, i]

    # UMAP
    umap_pre = emb_pre.get("umap2", None)
    if umap_pre is not None:
        df_pre["UMAP1_pre"] = umap_pre[:, 0]
        df_pre["UMAP2_pre"] = umap_pre[:, 1]

    # ---------- AFTER ----------
    df_post = meta_out.reset_index(drop=True).copy()

    pca_post = emb_post.get("pca20", None)
    if pca_post is not None:
        for i in range(pca_post.shape[1]):
            df_post[f"PCA{i+1}_post"] = pca_post[:, i]

    umap_post = emb_post.get("umap2", None)
    if umap_post is not None:
        df_post["UMAP1_post"] = umap_post[:, 0]
        df_post["UMAP2_post"] = umap_post[:, 1]

    # ---------- Combined (más cómodo) ----------
    df_combined = meta_out.reset_index(drop=True).copy()

    if pca_pre is not None:
        for i in range(pca_pre.shape[1]):
            df_combined[f"PCA{i+1}_pre"] = pca_pre[:, i]
    if pca_post is not None:
        for i in range(pca_post.shape[1]):
            df_combined[f"PCA{i+1}_post"] = pca_post[:, i]

    if umap_pre is not None:
        df_combined["UMAP1_pre"] = umap_pre[:, 0]
        df_combined["UMAP2_pre"] = umap_pre[:, 1]
    if umap_post is not None:
        df_combined["UMAP1_post"] = umap_post[:, 0]
        df_combined["UMAP2_post"] = umap_post[:, 1]

    # Guardar
    path_pre = out_dir / "qc_embeddings_before_combat.csv"
    path_post = out_dir / "qc_embeddings_after_combat.csv"
    path_combined = out_dir / "qc_embeddings_before_after_combat.csv"

    df_pre.to_csv(path_pre, index=False)
    df_post.to_csv(path_post, index=False)
    df_combined.to_csv(path_combined, index=False)

    print(f"  💾 {path_pre}")
    print(f"  💾 {path_post}")
    print(f"  💾 {path_combined}")

    return df_pre, df_post, df_combined

# =============================================================================
# EXTRA 2 – Validación con clasificadores sobre embeddings (PCA/UMAP)
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score


def _clean_target_for_clf(y, min_count_per_class=5):
    """
    Limpia target:
    - quita NaN/vacíos
    - quita clases con muy pocos casos
    """
    s = pd.Series(y).astype(str)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    mask = s.notna()
    s = s[mask]

    vc = s.value_counts()
    keep_classes = vc[vc >= min_count_per_class].index
    keep_mask = s.isin(keep_classes)

    return s[keep_mask], mask, keep_mask


def _eval_classifier_cv(X, y, random_state=42):
    """
    Evalúa un clasificador lineal (LogisticRegression) con CV estratificada.
    Devuelve métricas robustas multicategoría.
    """
    y = pd.Series(y).astype(str)
    n_classes = y.nunique()
    min_class = y.value_counts().min()

    # nº de folds seguro
    n_splits = int(min(5, max(2, min_class)))
    if n_splits < 2 or n_classes < 2 or len(y) < 20:
        return {
            "n_samples": len(y),
            "n_classes": int(n_classes),
            "cv_folds": np.nan,
            "acc": np.nan,
            "balanced_acc": np.nan,
            "macro_f1": np.nan
        }

    clf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs"
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Predicción out-of-fold
    y_pred = cross_val_predict(clf, X, y, cv=cv, method="predict")

    return {
        "n_samples": int(len(y)),
        "n_classes": int(n_classes),
        "cv_folds": int(n_splits),
        "acc": float(accuracy_score(y, y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y, y_pred)),
        "macro_f1": float(f1_score(y, y_pred, average="macro"))
    }


def _eval_one_target_on_embeddings(meta, target_col, emb_dict, emb_name, min_count_per_class=5):
    """
    Evalúa un target sobre PCA20 y UMAP2 (si existe) para un set de embeddings.
    """
    rows = []

    if target_col not in meta.columns:
        return rows

    # ---- PCA20 ----
    X_pca = emb_dict.get("pca20", None)
    if X_pca is not None:
        y_clean, mask_non_null, keep_mask = _clean_target_for_clf(meta[target_col], min_count_per_class=min_count_per_class)

        # reconstruir máscara completa sobre X
        idx_non_null = np.where(mask_non_null.values)[0]
        idx_keep = idx_non_null[keep_mask.values]

        X_use = X_pca[idx_keep]
        y_use = y_clean.values

        met = _eval_classifier_cv(X_use, y_use, random_state=42)
        met.update({
            "grouping": target_col,
            "embedding": "PCA20",
            "stage": emb_name
        })
        rows.append(met)

    # ---- UMAP2 ----
    X_umap = emb_dict.get("umap2", None)
    if X_umap is not None:
        y_clean, mask_non_null, keep_mask = _clean_target_for_clf(meta[target_col], min_count_per_class=min_count_per_class)

        idx_non_null = np.where(mask_non_null.values)[0]
        idx_keep = idx_non_null[keep_mask.values]

        X_use = X_umap[idx_keep]
        y_use = y_clean.values

        met = _eval_classifier_cv(X_use, y_use, random_state=42)
        met.update({
            "grouping": target_col,
            "embedding": "UMAP2",
            "stage": emb_name
        })
        rows.append(met)

    return rows


def run_embedding_classification_qc(metadata, emb_pre, emb_post, out_dir=OUTPUT_DIR, min_count_per_class=5):
    """
    Compara clasificabilidad de:
    - platform
    - dataset
    - label
    sobre embeddings BEFORE vs AFTER ComBat.

    Interpretación:
      * platform/dataset: idealmente BAJA la capacidad de clasificar
      * label: idealmente se mantiene o mejora
    """
    print("\n🧪 Evaluando clasificadores sobre embeddings (CV estratificada)...")

    meta = metadata.copy()

    rows = []
    for target in ["platform", "dataset", "label"]:
        rows += _eval_one_target_on_embeddings(meta, target, emb_pre,  "before", min_count_per_class=min_count_per_class)
        rows += _eval_one_target_on_embeddings(meta, target, emb_post, "after",  min_count_per_class=min_count_per_class)

    df = pd.DataFrame(rows)

    if df.empty:
        print("  ⚠️ No se pudieron calcular métricas de clasificación.")
        return df

    # Tabla resumen before vs after
    pivot_f1 = df.pivot_table(index=["grouping", "embedding"], columns="stage", values="macro_f1", aggfunc="first")
    pivot_ba = df.pivot_table(index=["grouping", "embedding"], columns="stage", values="balanced_acc", aggfunc="first")
    pivot_acc = df.pivot_table(index=["grouping", "embedding"], columns="stage", values="acc", aggfunc="first")

    summary = pd.DataFrame(index=pivot_f1.index).reset_index()
    summary["macro_f1_before"] = [pivot_f1.loc[idx, "before"] if "before" in pivot_f1.columns and idx in pivot_f1.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["macro_f1_after"]  = [pivot_f1.loc[idx, "after"]  if "after"  in pivot_f1.columns and idx in pivot_f1.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["delta_macro_f1"]  = summary["macro_f1_after"] - summary["macro_f1_before"]

    summary["balanced_acc_before"] = [pivot_ba.loc[idx, "before"] if "before" in pivot_ba.columns and idx in pivot_ba.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["balanced_acc_after"]  = [pivot_ba.loc[idx, "after"]  if "after"  in pivot_ba.columns and idx in pivot_ba.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["delta_balanced_acc"]  = summary["balanced_acc_after"] - summary["balanced_acc_before"]

    summary["acc_before"] = [pivot_acc.loc[idx, "before"] if "before" in pivot_acc.columns and idx in pivot_acc.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["acc_after"]  = [pivot_acc.loc[idx, "after"]  if "after"  in pivot_acc.columns and idx in pivot_acc.index else np.nan for idx in summary.set_index(["grouping","embedding"]).index]
    summary["delta_acc"]  = summary["acc_after"] - summary["acc_before"]

    # Añadimos n y clases (tomando info del BEFORE)
    aux = df[df["stage"] == "before"][["grouping", "embedding", "n_samples", "n_classes", "cv_folds"]].drop_duplicates()
    summary = summary.merge(aux, on=["grouping", "embedding"], how="left")

    # Guardar
    path_raw = out_dir / "qc_embedding_classification_raw.csv"
    path_summary = out_dir / "qc_embedding_classification_summary.csv"
    df.to_csv(path_raw, index=False)
    summary.to_csv(path_summary, index=False)

    print(f"  💾 {path_raw}")
    print(f"  💾 {path_summary}")

    print("\n  Resumen clasificación (before vs after):")
    print(summary.to_string(index=False))

    print("\n  Interpretación:")
    print("   - platform/dataset: idealmente BAJAN (más difícil distinguir batch)")
    print("   - label: idealmente se mantiene o SUBE (señal biológica preservada)")

    return summary

    

# =============================================================================
# PASO 1 – CARGA
# =============================================================================

def load_microarrays():
    print("\n📦 [1/3] Cargando microarrays...")

    expr = pd.read_csv(MICRO_MATRIX, index_col=0)
    meta = pd.read_csv(MICRO_METADATA)
    meta = clean_columns(meta)

    expr.index = expr.index.astype(str).str.strip()
    expr = ensure_unique_index(expr, axis_name="genes")

    print(f"  Matriz: {expr.shape[0]} genes x {expr.shape[1]} muestras")

    expr = detect_and_log2(expr, "Microarrays")
    expr, meta = align_samples(expr, meta, MICRO_SAMPLE_COL, "Microarrays")

    # Mapeo GSE -> plataforma (añade aquí los que falten)
    platform_map = {
        "GSE1456":   "GPL96",
        "GSE15852":  "GPL96",
        "GSE25066":  "GPL96",
        "GSE162228": "GPL570",
        "GSE19615":  "GPL570",
        "GSE20711":  "GPL570",
        "GSE21653":  "GPL570",
        "GSE32646":  "GPL570",
        "GSE58812":  "GPL570",
        "GSE65194":  "GPL570",
        "GSE70947": "GPL13607",
    }

    if MICRO_GSE_COL not in meta.columns:
        raise ValueError(f"[Microarrays] No existe '{MICRO_GSE_COL}' en metadata.")

    meta["dataset"]   = meta[MICRO_GSE_COL].astype(str).str.strip()
    meta["platform"]  = meta["dataset"].map(platform_map).fillna("unknown")
    meta["data_type"] = "microarray"

    print(f"  Muestras por GSE:\n{meta['dataset'].value_counts().to_string()}")
    return expr, meta


def load_tcga():
    print("\n📦 [2/3] Cargando TCGA-BRCA...")

    expr = pd.read_csv(TCGA_MATRIX, index_col=0)
    meta = pd.read_csv(TCGA_METADATA)
    meta = clean_columns(meta)

    expr.index = expr.index.astype(str).str.strip()
    expr = ensure_unique_index(expr, axis_name="genes")

    print(f"  Matriz: {expr.shape[0]} genes x {expr.shape[1]} muestras")

    if TCGA_TYPE in ("counts", "fpkm"):
        print(f"  Aplicando log2(x+1) a {TCGA_TYPE}...")
        expr = np.log2(expr.clip(lower=0) + 1)
    else:
        print("  Asumiendo que ya está en log2.")

    if len(expr.index) > 0 and str(expr.index[0]).startswith("ENSG"):
        print("  ENSEMBL IDs detectados → limpiando versiones (.X)...")
        expr.index = expr.index.str.split(".").str[0]
        expr = ensure_unique_index(expr, axis_name="genes (ENSG limpiados)")

    expr, meta = align_samples(expr, meta, TCGA_SAMPLE_COL, "TCGA-BRCA")

    meta["dataset"]   = "TCGA-BRCA"
    meta["platform"]  = "RNAseq"
    meta["data_type"] = "rnaseq"

    print(f"  TCGA-BRCA: {expr.shape[1]} muestras")
    return expr, meta


def load_gse96058():
    print("\n📦 [3/3] Cargando GSE96058...")

    expr = pd.read_csv(GSE96058_MATRIX, index_col=0)
    meta = pd.read_csv(GSE96058_METADATA)
    meta = clean_columns(meta)

    expr.index = expr.index.astype(str).str.strip()
    expr = ensure_unique_index(expr, axis_name="genes")

    print(f"  Matriz: {expr.shape[0]} genes x {expr.shape[1]} muestras")

    expr = detect_and_log2(expr, "GSE96058")
    expr, meta = align_samples(expr, meta, GSE96058_SAMPLE_COL, "GSE96058")

    meta["dataset"]   = "GSE96058"
    meta["platform"]  = "RNAseq"
    meta["data_type"] = "rnaseq"

    print(f"  GSE96058: {expr.shape[1]} muestras")
    return expr, meta


# =============================================================================
# PASO 2 – INTEGRACIÓN
# =============================================================================

def integrate(micro_expr, micro_meta, tcga_expr, tcga_meta, gse_expr, gse_meta):
    print("\n🔗 Integrando datasets...")

    for _df in (micro_expr, tcga_expr, gse_expr):
        _df.index = _df.index.astype(str).str.strip()

    genes_common = sorted(set(micro_expr.index) & set(tcga_expr.index) & set(gse_expr.index))
    print(f"  Genes comunes: {len(genes_common)}")
    if len(genes_common) < 500:
        print("  ⚠️ Muy pocos genes comunes. Verifica Gene Symbol en todas las matrices.")

    expr_combined = pd.concat([
        micro_expr.reindex(genes_common),
        tcga_expr.reindex(genes_common),
        gse_expr.reindex(genes_common),
    ], axis=1)

    meta_combined = pd.concat([micro_meta, tcga_meta, gse_meta], axis=0, join="outer")
    meta_combined = meta_combined.reindex(expr_combined.columns)

    na_pct = expr_combined.isna().mean().mean() * 100
    if na_pct > 0:
        print(f"  Imputando {na_pct:.3f}% de NAs con mediana por gen...")
        row_medians = expr_combined.median(axis=1)
        expr_combined = expr_combined.T.fillna(row_medians).T

    print(f"  Matriz final: {expr_combined.shape[0]} genes x {expr_combined.shape[1]} muestras")
    print(f"  Muestras por plataforma:\n{meta_combined['platform'].value_counts(dropna=False).to_string()}")
    print(f"  Muestras por dataset:\n{meta_combined['dataset'].value_counts(dropna=False).to_string()}")

    return expr_combined, meta_combined


# =============================================================================
# PASO 3 – ComBat
# =============================================================================

def apply_combat(expr: pd.DataFrame,
                 metadata: pd.DataFrame,
                 batch_col: str = "dataset",
                 protect_col: str = None) -> pd.DataFrame:
    try:
        from inmoose.pycombat import pycombat_norm
    except ImportError:
        raise ImportError("Instala inmoose: pip install inmoose")

    print(f"\n⚙️  Aplicando ComBat (batch = '{batch_col}')...")

    meta = metadata.reindex(expr.columns).copy()

    if batch_col not in meta.columns:
        raise ValueError(f"No existe batch_col='{batch_col}' en metadata.")

    meta[batch_col] = meta[batch_col].astype(str).str.strip()
    meta.loc[meta[batch_col] == "", batch_col] = "NA_BATCH"
    batch = meta[batch_col].values

    covar_mod = None
    if protect_col is not None:
        if protect_col not in meta.columns:
            print(f"  ⚠️ protect_col='{protect_col}' no existe en metadata. Se ignora.")
        else:
            cov = meta[protect_col].astype(str).str.strip()
            cov = cov.replace({"": np.nan, "nan": np.nan, "None": np.nan})
            if cov.notna().sum() > 0 and cov.nunique(dropna=True) >= 2:
                covar_mod = pd.get_dummies(cov, drop_first=True).astype(float)
                covar_mod.index = expr.columns
                print(f"  Protegiendo covariable: '{protect_col}' ({cov.nunique(dropna=True)} niveles)")
            else:
                print(f"  ⚠️ protect_col='{protect_col}' sin variabilidad suficiente. Se ignora.")

    # filtrar genes con varianza casi nula
    gene_var = expr.var(axis=1)
    keep = gene_var[gene_var > 1e-12].index
    expr_use = expr.loc[keep].copy()

    # IMPORTANTE: 'counts', no 'data'
    corrected = pycombat_norm(counts=expr_use, batch=batch, covar_mod=covar_mod)
    corrected = pd.DataFrame(corrected, index=expr_use.index, columns=expr_use.columns)

    # restaurar genes filtrados para mantener dimensiones
    if len(keep) < expr.shape[0]:
        removed = expr.index.difference(keep)
        corrected = pd.concat([corrected, expr.loc[removed]], axis=0).loc[expr.index]

    print("  ✅ ComBat completado.")
    return corrected


# =============================================================================
# PASO 4 – QC (PCA + UMAP + silhouette)
# =============================================================================

def _make_color_map(categories):
    categories = [str(c) for c in categories]
    if len(categories) <= 10:
        pal = sns.color_palette("tab10", n_colors=len(categories))
    else:
        pal = sns.color_palette("tab20", n_colors=len(categories))
    return dict(zip(categories, pal))


def _scatter_with_fixed_colors(ax, coords, meta, col, title):
    labels = meta[col].astype(str).fillna("NA")
    cats = sorted(labels.unique().tolist())
    cmap = _make_color_map(cats)
    colors = [cmap[str(v)] for v in labels.values]

    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.65, linewidths=0)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Dim1")
    ax.set_ylabel("Dim2")
    ax.grid(alpha=0.2)

    if len(cats) <= 20:
        handles = [
            plt.Line2D([0], [0], marker='o', linestyle='', markersize=5,
                       markerfacecolor=cmap[c], markeredgecolor='none', label=c)
            for c in cats
        ]
        ax.legend(handles=handles, title=col, fontsize=7, frameon=False, loc="best")
    else:
        ax.text(0.01, 0.99, f"{len(cats)} categorías", transform=ax.transAxes,
                va="top", ha="left", fontsize=8,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))


def _prepare_embeddings(expr: pd.DataFrame):
    """
    expr: genes x muestras
    Devuelve PCA coords (2D y 20D) y UMAP 2D (si disponible)
    """
    X = expr.T.fillna(0).values  # muestras x genes
    Xz = StandardScaler().fit_transform(X)

    n_comp = min(PCA_N_COMPONENTS, Xz.shape[0]-1, Xz.shape[1]-1)
    n_comp = max(2, n_comp)

    pca_model = PCA(n_components=n_comp, random_state=UMAP_RANDOM_STATE)
    X_pca = pca_model.fit_transform(Xz)

    out = {
        "pca2": X_pca[:, :2],
        "pca20": X_pca[:, :min(20, X_pca.shape[1])]
    }

    if HAS_UMAP:
        try:
            reducer = umap.UMAP(
                n_neighbors=UMAP_N_NEIGHBORS,
                min_dist=UMAP_MIN_DIST,
                n_components=2,
                metric="euclidean",
                random_state=UMAP_RANDOM_STATE
            )
            # UMAP sobre PCs (más estable/rápido)
            X_umap = reducer.fit_transform(X_pca[:, :min(30, X_pca.shape[1])])
            out["umap2"] = X_umap
        except Exception as e:
            print(f"  ⚠️ UMAP no se pudo calcular: {e}")
            out["umap2"] = None
    else:
        print("  ⚠️ UMAP no disponible (instala umap-learn)")
        out["umap2"] = None

    return out


def _safe_silhouette(X, labels):
    if X is None:
        return np.nan

    s = pd.Series(labels).astype(str)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    mask = s.notna().values
    s = s[mask]
    Xf = X[mask]

    if len(s) < 10 or s.nunique() < 2:
        return np.nan

    vc = s.value_counts()
    valid_cats = vc[vc >= 2].index
    keep = s.isin(valid_cats).values
    s = s[keep]
    Xf = Xf[keep]

    if len(s) < 10 or s.nunique() < 2:
        return np.nan

    try:
        return float(silhouette_score(Xf, s.values))
    except Exception:
        return np.nan


def compute_silhouette_report(meta, emb_pre, emb_post):
    rows = []
    for col in ["platform", "dataset", "label"]:
        if col not in meta.columns:
            continue

        rows.append({
            "grouping": col,
            "silhouette_pre_pca": _safe_silhouette(emb_pre["pca20"], meta[col]),
            "silhouette_post_pca": _safe_silhouette(emb_post["pca20"], meta[col]),
            "silhouette_pre_umap": _safe_silhouette(emb_pre["umap2"], meta[col]),
            "silhouette_post_umap": _safe_silhouette(emb_post["umap2"], meta[col]),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df["delta_pca"] = df["silhouette_post_pca"] - df["silhouette_pre_pca"]
        df["delta_umap"] = df["silhouette_post_umap"] - df["silhouette_pre_umap"]

    out_csv = OUTPUT_DIR / "qc_silhouette_before_after_combat.csv"
    df.to_csv(out_csv, index=False)
    print(f"  💾 {out_csv}")
    return df


def plot_density(expr, metadata, title, ax, max_points_per_dataset=200000):
    meta = metadata.reindex(expr.columns).copy()
    meta["dataset"] = meta["dataset"].astype(str).fillna("NA")

    datasets = sorted(meta["dataset"].unique().tolist())
    pal = sns.color_palette("tab20", n_colors=max(1, len(datasets)))
    cmap = dict(zip(datasets, pal))

    rng = np.random.default_rng(42)

    for ds, grp in meta.groupby("dataset", dropna=False):
        samples = [s for s in grp.index if s in expr.columns]
        if len(samples) == 0:
            continue
        vals = expr[samples].values.flatten()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        # downsample para acelerar KDE
        if len(vals) > max_points_per_dataset:
            idx = rng.choice(len(vals), size=max_points_per_dataset, replace=False)
            vals = vals[idx]

        sns.kdeplot(vals, ax=ax, label=ds, color=cmap.get(ds, (0.5, 0.5, 0.5)), linewidth=1.1)

    ax.set_xlabel("Expresión (log2)")
    ax.set_ylabel("Densidad")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.legend(fontsize=6, ncol=2, frameon=False)


def save_qc_plots(expr_raw, expr_corrected, metadata):
    print("\n📊 Generando gráficas de diagnóstico (PCA/UMAP/silhouette)...")

    meta = metadata.reindex(expr_raw.columns).copy()

    # Embeddings
    print("  · Calculando embeddings BEFORE...")
    emb_pre = _prepare_embeddings(expr_raw)

    print("  · Calculando embeddings AFTER...")
    emb_post = _prepare_embeddings(expr_corrected)

    # Figura 1: platform (PCA/UMAP) + densidad
    fig, axes = plt.subplots(3, 2, figsize=(14, 16))

    _scatter_with_fixed_colors(axes[0, 0], emb_pre["pca2"],  meta, "platform", "PCA BEFORE · color=platform")
    _scatter_with_fixed_colors(axes[0, 1], emb_post["pca2"], meta, "platform", "PCA AFTER · color=platform")

    if emb_pre["umap2"] is not None and emb_post["umap2"] is not None:
        _scatter_with_fixed_colors(axes[1, 0], emb_pre["umap2"],  meta, "platform", "UMAP BEFORE · color=platform")
        _scatter_with_fixed_colors(axes[1, 1], emb_post["umap2"], meta, "platform", "UMAP AFTER · color=platform")
    else:
        axes[1, 0].text(0.5, 0.5, "UMAP no disponible", ha="center", va="center")
        axes[1, 1].text(0.5, 0.5, "UMAP no disponible", ha="center", va="center")
        axes[1, 0].set_axis_off()
        axes[1, 1].set_axis_off()

    plot_density(expr_raw,       meta, "Densidad BEFORE", axes[2, 0])
    plot_density(expr_corrected, meta, "Densidad AFTER",  axes[2, 1])

    plt.suptitle("QC integración multi-plataforma · BEFORE vs AFTER ComBat", fontsize=14, y=0.995)
    plt.tight_layout()
    fig1_path = OUTPUT_DIR / "diagnostico_combat_pca_umap_density.png"
    plt.savefig(fig1_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  💾 {fig1_path}")

    # Figura 2: label (PCA/UMAP), si existe
    if "label" in meta.columns:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        _scatter_with_fixed_colors(axes[0, 0], emb_pre["pca2"],  meta, "label", "PCA BEFORE · color=label")
        _scatter_with_fixed_colors(axes[0, 1], emb_post["pca2"], meta, "label", "PCA AFTER · color=label")

        if emb_pre["umap2"] is not None and emb_post["umap2"] is not None:
            _scatter_with_fixed_colors(axes[1, 0], emb_pre["umap2"],  meta, "label", "UMAP BEFORE · color=label")
            _scatter_with_fixed_colors(axes[1, 1], emb_post["umap2"], meta, "label", "UMAP AFTER · color=label")
        else:
            axes[1, 0].text(0.5, 0.5, "UMAP no disponible", ha="center", va="center")
            axes[1, 1].text(0.5, 0.5, "UMAP no disponible", ha="center", va="center")
            axes[1, 0].set_axis_off()
            axes[1, 1].set_axis_off()

        plt.suptitle("QC por label · BEFORE vs AFTER ComBat", fontsize=13, y=0.995)
        plt.tight_layout()
        fig2_path = OUTPUT_DIR / "diagnostico_combat_label_pca_umap.png"
        plt.savefig(fig2_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  💾 {fig2_path}")

    # Silhouette
    sil_df = compute_silhouette_report(meta, emb_pre, emb_post)
    print("\n  Silhouette (interpreta así):")
    print("   - platform/dataset: idealmente BAJAN tras ComBat")
    print("   - label: idealmente se mantiene (no cae mucho)")
    if not sil_df.empty:
        print(sil_df.to_string(index=False))

    return emb_pre, emb_post, sil_df


# =============================================================================
# EJECUCIÓN
# =============================================================================

def run():
    print("=" * 60)
    print("  INTEGRACIÓN MULTI-PLATAFORMA + ComBat – CÁNCER DE MAMA")
    print("=" * 60)

    # Carga
    micro_expr, micro_meta = load_microarrays()
    tcga_expr,  tcga_meta  = load_tcga()
    gse_expr,   gse_meta   = load_gse96058()

    # Integración
    expr_raw, metadata = integrate(
        micro_expr, micro_meta,
        tcga_expr,  tcga_meta,
        gse_expr,   gse_meta,
    )

    expr_raw.to_csv(OUTPUT_DIR / "expr_raw_combined.csv")
    metadata.to_csv(OUTPUT_DIR / "metadata_combined.csv")
    print("  💾 expr_raw_combined.csv")
    print("  💾 metadata_combined.csv")

    # ComBat
    expr_combat = apply_combat(
        expr=expr_raw,
        metadata=metadata,
        batch_col=COMBAT_BATCH_COL,
        protect_col=PROTECT_COL,
    )
    expr_combat.to_csv(OUTPUT_DIR / "expr_combat_corrected.csv")
    print("  💾 expr_combat_corrected.csv")


    # QC embeddings + silhouette
    emb_pre, emb_post, sil_df = save_qc_plots(expr_raw, expr_combat, metadata)
    
    # Guardar coordenadas PCA/UMAP por muestra
    meta_qc = metadata.reindex(expr_raw.columns).copy()
    save_embeddings_csv(meta_qc, emb_pre, emb_post, out_dir=OUTPUT_DIR)
    
    # Clasificadores before vs after sobre embeddings
    run_embedding_classification_qc(meta_qc, emb_pre, emb_post, out_dir=OUTPUT_DIR, min_count_per_class=5)

    print("\n" + "=" * 60)
    print("  ✅ COMPLETADO")
    print(f"  📁 {OUTPUT_DIR.resolve()}")
    print("=" * 60)


if __name__ == "__main__":
    run()

  INTEGRACIÓN MULTI-PLATAFORMA + ComBat – CÁNCER DE MAMA

📦 [1/3] Cargando microarrays...
  Matriz: 13236 genes x 1499 muestras
  [Microarrays] Escala log2 detectada (max p99=12.6) → sin transformación
  Muestras por GSE:
dataset
GSE25066     508
GSE21653     266
GSE65194     164
GSE162228    133
GSE58812     107
GSE1456      102
GSE20711      90
GSE32646      44
GSE15852      43
GSE19615      42

📦 [2/3] Cargando TCGA-BRCA...
  Matriz: 20530 genes x 904 muestras
  Asumiendo que ya está en log2.
  TCGA-BRCA: 904 muestras

📦 [3/3] Cargando GSE96058...
  Matriz: 30865 genes x 3409 muestras
  [GSE96058] Escala log2 detectada (max p99=7.9) → sin transformación
  GSE96058: 3409 muestras

🔗 Integrando datasets...
  Genes comunes: 11907
  Matriz final: 11907 genes x 5812 muestras
  Muestras por plataforma:
platform
RNAseq    4313
GPL570     846
GPL96      653
  Muestras por dataset:
dataset
GSE96058     3409
TCGA-BRCA     904
GSE25066      508
GSE21653      266
GSE65194      164
GSE162228    